In [ ]:
"""Grazioso Salvare interactive animal rescue dashboard."""

import logging

import dash_leaflet as dl
import pandas as pd
import plotly.express as px
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash

from animal_shelter import AnimalShelter
from config import (
    MONGO_COLLECTION,
    MONGO_DATABASE,
    MONGO_HOST,
    MONGO_PASSWORD,
    MONGO_PORT,
    MONGO_USERNAME,
)


# Configure structured application logging.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

LOGGER = logging.getLogger("grazioso_dashboard")


# Rescue classification rules are stored in one location so they can be
# maintained without rewriting the dashboard callback.
RESCUE_FILTERS = {
    "water": {
        "breed": {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland",
            ]
        },
        "sex_upon_outcome": "Intact Female",
    },
    "mountain": {
        "breed": {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler",
            ]
        },
        "sex_upon_outcome": "Intact Male",
    },
    "disaster": {
        "breed": {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler",
            ]
        },
        "sex_upon_outcome": "Intact Male",
    },
    "reset": {},
}


def get_rescue_query(filter_type):
    """Return the MongoDB query associated with a rescue filter."""

    if filter_type not in RESCUE_FILTERS:
        LOGGER.warning(
            "Unknown filter '%s' received. Resetting dashboard.",
            filter_type,
        )
        return {}

    return RESCUE_FILTERS[filter_type]


def load_animal_records(query=None):
    """Retrieve animal records and convert them into a DataFrame."""

    if query is None:
        query = {}

    try:
        records = shelter.read(query)
        return pd.DataFrame.from_records(records)

    except (RuntimeError, ValueError, TypeError) as error:
        LOGGER.error("Unable to load animal records: %s", error)
        return pd.DataFrame()


def create_empty_message(message):
    """Create a user-friendly dashboard message."""

    return html.Div(
        message,
        style={
            "textAlign": "center",
            "padding": "30px",
            "fontWeight": "bold",
        },
    )


# Establish the database connection through the enhanced data access class.
try:
    shelter = AnimalShelter(
        username=MONGO_USERNAME,
        password=MONGO_PASSWORD,
        host=MONGO_HOST,
        port=MONGO_PORT,
        database_name=MONGO_DATABASE,
        collection_name=MONGO_COLLECTION,
    )

    df = load_animal_records({})
    LOGGER.info("Initial dashboard dataset contains %d record(s).", len(df))

except (ConnectionError, RuntimeError, ValueError) as error:
    LOGGER.critical("Dashboard startup failed: %s", error)
    shelter = None
    df = pd.DataFrame()


app = JupyterDash("ProjectTwo")


app.layout = html.Div(
    [
        html.Center(
            html.B(
                html.H1("Grazioso Salvare Dashboard")
            )
        ),
        html.Center(
            html.H4("Praise Oulare - CS 499 Software Engineering Enhancement")
        ),
        html.Hr(),

        html.Div(
            [
                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {
                            "label": "Water Rescue",
                            "value": "water",
                        },
                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "mountain",
                        },
                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "disaster",
                        },
                        {
                            "label": "Reset (Show All)",
                            "value": "reset",
                        },
                    ],
                    value="reset",
                    labelStyle={
                        "display": "inline-block",
                        "padding": "10px",
                    },
                )
            ],
            style={"textAlign": "center"},
        ),

        html.Hr(),

        html.Div(
            id="dashboard-status",
            style={
                "textAlign": "center",
                "paddingBottom": "10px",
                "fontWeight": "bold",
            },
        ),

        dash_table.DataTable(
            id="datatable-id",
            columns=[
                {
                    "name": column_name,
                    "id": column_name,
                    "deletable": False,
                    "selectable": True,
                }
                for column_name in df.columns
            ],
            data=df.to_dict("records"),
            page_size=10,
            sort_action="native",
            filter_action="native",
            row_selectable="single",
            selected_rows=[0] if not df.empty else [],
            style_table={
                "overflowX": "auto",
            },
            style_cell={
                "textAlign": "left",
                "minWidth": "100px",
                "maxWidth": "250px",
                "whiteSpace": "normal",
            },
        ),

        html.Br(),

        html.Div(
            className="row",
            children=[
                html.Div(
                    id="graph-id",
                    className="col s12 m6",
                    style={
                        "width": "50%",
                        "display": "inline-block",
                        "verticalAlign": "top",
                    },
                ),
                html.Div(
                    id="map-id",
                    className="col s12 m6",
                    style={
                        "width": "50%",
                        "display": "inline-block",
                        "verticalAlign": "top",
                    },
                ),
            ],
        ),
    ]
)


@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("datatable-id", "selected_rows"),
        Output("dashboard-status", "children"),
    ],
    [Input("filter-type", "value")],
)
def update_dashboard(filter_type):
    """Update the table when the user selects a rescue category."""

    if shelter is None:
        return (
            [],
            [],
            [],
            "Database connection is unavailable.",
        )

    query = get_rescue_query(filter_type)
    filtered_df = load_animal_records(query)

    LOGGER.info(
        "Filter '%s' returned %d record(s).",
        filter_type,
        len(filtered_df),
    )

    if filtered_df.empty:
        return (
            [],
            [],
            [],
            "No animal records matched the selected rescue category.",
        )

    columns = [
        {
            "name": column_name,
            "id": column_name,
            "deletable": False,
            "selectable": True,
        }
        for column_name in filtered_df.columns
    ]

    status_message = (
        f"Displaying {len(filtered_df)} animal record(s)."
    )

    return (
        filtered_df.to_dict("records"),
        columns,
        [0],
        status_message,
    )


@app.callback(
    [
        Output("map-id", "children"),
        Output("graph-id", "children"),
    ],
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map_and_graph(view_data, selected_rows):
    """Update the map and breed chart using the visible table records."""

    if not view_data:
        empty_message = create_empty_message(
            "No data is available for visualization."
        )
        return empty_message, empty_message

    visible_df = pd.DataFrame.from_records(view_data)

    if visible_df.empty:
        empty_message = create_empty_message(
            "No data is available for visualization."
        )
        return empty_message, empty_message

    selected_index = 0

    if selected_rows:
        selected_index = selected_rows[0]

    if selected_index >= len(visible_df):
        selected_index = 0

    required_map_columns = {
        "location_lat",
        "location_long",
    }

    if required_map_columns.issubset(visible_df.columns):
        selected_animal = visible_df.iloc[selected_index]

        latitude = selected_animal.get("location_lat")
        longitude = selected_animal.get("location_long")
        animal_name = selected_animal.get("name", "Unknown")
        animal_breed = selected_animal.get("breed", "Unknown")

        if pd.notna(latitude) and pd.notna(longitude):
            map_view = dl.Map(
                style={
                    "width": "100%",
                    "height": "500px",
                },
                center=[latitude, longitude],
                zoom=10,
                children=[
                    dl.TileLayer(id="base-layer-id"),
                    dl.Marker(
                        position=[latitude, longitude],
                        children=[
                            dl.Tooltip(str(animal_breed)),
                            dl.Popup(
                                [
                                    html.H3("Animal Information"),
                                    html.P(f"Name: {animal_name}"),
                                    html.P(f"Breed: {animal_breed}"),
                                ]
                            ),
                        ],
                    ),
                ],
            )
        else:
            map_view = create_empty_message(
                "The selected animal does not have valid location data."
            )
    else:
        map_view = create_empty_message(
            "Location columns are unavailable in the current dataset."
        )

    if "breed" in visible_df.columns:
        breed_counts = (
            visible_df["breed"]
            .fillna("Unknown")
            .value_counts()
            .reset_index()
        )

        breed_counts.columns = [
            "breed",
            "count",
        ]

        figure = px.pie(
            breed_counts,
            names="breed",
            values="count",
            title="Breed Distribution",
        )

        graph_view = dcc.Graph(figure=figure)
    else:
        graph_view = create_empty_message(
            "Breed information is unavailable."
        )

    return map_view, graph_view


if __name__ == "__main__":
    app.run_server(mode="inline")